In [1]:
import json
import pandas as pd

with open("transactions.json", "r") as f:
    transactions = json.load(f)

df = pd.DataFrame(transactions)

print(df.head())
print(df.shape)

         date                     description    category  amount    type
0  2026-03-01  Client Payment - Project Alpha     Revenue   45000  Credit
1  2026-03-02    Vendor: Intermountain Lumber   Inventory  -12000   Debit
2  2026-03-04                 Payroll Service  Operations   -8500   Debit
3  2026-03-05   Client Payment - Project Beta     Revenue   22000  Credit
4  2026-03-08       Lease Payment - Excavator   Equipment   -2200   Debit
(20, 5)


In [2]:
df["date"] = pd.to_datetime(df["date"])

print(df["date"].min())
print(df["date"].max())

2026-03-01 00:00:00
2026-03-31 00:00:00


In [3]:
print(df.isnull().sum())

date           0
description    0
category       0
amount         0
type           0
dtype: int64


In [4]:
print(df.duplicated().sum())

0


In [5]:
invalid_credit = df[
    (df["type"] == "Credit") &
    (df["amount"] <= 0)
]

invalid_debit = df[
    (df["type"] == "Debit") &
    (df["amount"] >= 0)
]

print(invalid_credit)
print(invalid_debit)

Empty DataFrame
Columns: [date, description, category, amount, type]
Index: []
Empty DataFrame
Columns: [date, description, category, amount, type]
Index: []


In [6]:

print("\nTransaction Type Count:")
print(df["type"].value_counts())



print("\nTransaction Count by Category:")
print(df["category"].value_counts())



category_summary = (
    df.groupby("category")
      .agg(
          transaction_count=("amount", "count"),
          net_amount=("amount", "sum")
      )
      .sort_values("net_amount", ascending=False)
)

print("\nCategory Summary:")
print(category_summary)


Transaction Type Count:
type
Debit     15
Credit     5
Name: count, dtype: int64

Transaction Count by Category:
category
Operations     6
Revenue        4
Inventory      4
Tech/Growth    2
Equipment      1
Interest       1
Savings        1
Growth         1
Name: count, dtype: int64

Category Summary:
             transaction_count  net_amount
category                                  
Revenue                      4      112000
Interest                     1         120
Tech/Growth                  2        -650
Equipment                    1       -2200
Savings                      1       -5000
Growth                       1      -10000
Operations                   6      -18750
Inventory                    4      -30800


In [7]:

category_mapping = {
    "Revenue": "Revenue",
    "Interest": "Other Income",
    "Inventory": "Material Expense",
    "Operations": "Operating Expense",
    "Equipment": "Operating Expense",
    "Tech/Growth": "Growth Investment",
    "Growth": "Growth Investment",
    "Savings": "Internal Transfer"
}


df["economic_bucket"] = df["category"].map(category_mapping)

print("\nTransactions with Economic Classification:")
print(
    df[
        ["date", "description", "category",
         "amount", "economic_bucket"]
    ]
)


economic_summary = (
    df.groupby("economic_bucket")
      .agg(
          transaction_count=("amount", "count"),
          net_amount=("amount", "sum")
      )
      .sort_values("net_amount", ascending=False)
)

print("\nEconomic Summary:")
print(economic_summary)


Transactions with Economic Classification:
         date                      description     category  amount  \
0  2026-03-01   Client Payment - Project Alpha      Revenue   45000   
1  2026-03-02     Vendor: Intermountain Lumber    Inventory  -12000   
2  2026-03-04                  Payroll Service   Operations   -8500   
3  2026-03-05    Client Payment - Project Beta      Revenue   22000   
4  2026-03-08        Lease Payment - Excavator    Equipment   -2200   
5  2026-03-10                       Gas & Fuel   Operations    -600   
6  2026-03-12          Vendor: Steel Supply Co    Inventory   -4000   
7  2026-03-15         Client Payment - Deposit      Revenue   15000   
8  2026-03-18  Software Subscription (AutoCAD)  Tech/Growth    -150   
9  2026-03-20            Marketing - Local SEO  Tech/Growth    -500   
10 2026-03-22            Bonus Interest Earned     Interest     120   
11 2026-03-23     Vendor: Intermountain Lumber    Inventory  -11000   
12 2026-03-25                  Pa

In [8]:

revenue = df.loc[
    df["economic_bucket"] == "Revenue", "amount"
].sum()

other_income = df.loc[
    df["economic_bucket"] == "Other Income", "amount"
].sum()

material_expense = abs(
    df.loc[
        df["economic_bucket"] == "Material Expense", "amount"
    ].sum()
)

operating_expense = abs(
    df.loc[
        df["economic_bucket"] == "Operating Expense", "amount"
    ].sum()
)

growth_investment = abs(
    df.loc[
        df["economic_bucket"] == "Growth Investment", "amount"
    ].sum()
)

internal_transfer = abs(
    df.loc[
        df["economic_bucket"] == "Internal Transfer", "amount"
    ].sum()
)


core_operating_surplus = (
    revenue
    - material_expense
    - operating_expense
)


total_credits = df.loc[
    df["type"] == "Credit", "amount"
].sum()

total_debits = abs(
    df.loc[
        df["type"] == "Debit", "amount"
    ].sum()
)

net_cash_movement = total_credits - total_debits


# Ratios
material_to_revenue = material_expense / revenue
operating_to_revenue = operating_expense / revenue
growth_to_revenue = growth_investment / revenue


print("\nCore Financial Metrics")
print("----------------------")

print(f"Revenue: ${revenue:,.2f}")
print(f"Other Income: ${other_income:,.2f}")

print(f"Material Expense: ${material_expense:,.2f}")
print(f"Operating Expense: ${operating_expense:,.2f}")
print(f"Growth Investment: ${growth_investment:,.2f}")
print(f"Internal Transfer: ${internal_transfer:,.2f}")

print(f"\nCore Operating Cash Surplus: ${core_operating_surplus:,.2f}")
print(f"Net Observed Cash Movement: ${net_cash_movement:,.2f}")

print(f"\nMaterial / Revenue: {material_to_revenue:.2%}")
print(f"Operating Expense / Revenue: {operating_to_revenue:.2%}")
print(f"Growth Investment / Revenue: {growth_to_revenue:.2%}")


Core Financial Metrics
----------------------
Revenue: $112,000.00
Other Income: $120.00
Material Expense: $30,800.00
Operating Expense: $20,950.00
Growth Investment: $10,650.00
Internal Transfer: $5,000.00

Core Operating Cash Surplus: $60,250.00
Net Observed Cash Movement: $44,720.00

Material / Revenue: 27.50%
Operating Expense / Revenue: 18.71%
Growth Investment / Revenue: 9.51%


In [10]:

material_burden_pct = (
    material_expense / revenue
) * 100


core_cash_retention_pct = (
    core_operating_surplus / revenue
) * 100


positive_cash_flag = net_cash_movement > 0


growth_investment_flag = growth_investment > 0



resilience_alpha = {
    "signal_name": "Resilience Alpha",

    "market_context": {
        "industry_material_cost_inflation_pct": 15,
        "industry_average_net_margin_pct": 4
    },

    "observed_metrics": {
    "revenue": int(revenue),
    "material_expense": int(material_expense),
    "material_burden_pct": round(float(material_burden_pct), 2),
    "core_operating_cash_surplus": int(core_operating_surplus),
    "core_cash_retention_pct": round(float(core_cash_retention_pct), 2),
    "net_cash_movement": int(net_cash_movement),
    "growth_investment": int(growth_investment)
    },

    "positive_signal": (
        positive_cash_flag and growth_investment_flag
    ),

    "interpretation": (
        "Despite industry-wide material cost pressure, "
        "Elite Builds shows positive observed cash generation "
        "while continuing to invest in growth."
    ),

    "limitation": (
        "The dataset contains only the supplied March 2026 "
        "transactions and does not provide a historical "
        "company-level material-cost baseline."
    )
}


print("\nResilience Alpha Signal")
print("-----------------------")

for key, value in resilience_alpha.items():
    print(f"{key}: {value}")


Resilience Alpha Signal
-----------------------
signal_name: Resilience Alpha
market_context: {'industry_material_cost_inflation_pct': 15, 'industry_average_net_margin_pct': 4}
observed_metrics: {'revenue': 112000, 'material_expense': 30800, 'material_burden_pct': 27.5, 'core_operating_cash_surplus': 60250, 'core_cash_retention_pct': 53.79, 'net_cash_movement': 44720, 'growth_investment': 10650}
positive_signal: True
interpretation: Despite industry-wide material cost pressure, Elite Builds shows positive observed cash generation while continuing to invest in growth.
limitation: The dataset contains only the supplied March 2026 transactions and does not provide a historical company-level material-cost baseline.


In [11]:
# ---------------------------------
# STEP 7: GROWTH REINVESTMENT SIGNAL
# ---------------------------------

# Business outflows exclude internal savings transfer
business_outflows = (
    material_expense
    + operating_expense
    + growth_investment
)

# Growth investment relative to revenue
growth_to_revenue_pct = (
    growth_investment / revenue
) * 100

# Share of business outflows going toward growth
growth_outflow_share_pct = (
    growth_investment / business_outflows
) * 100

# Growth spending compared with routine operating + growth spending
growth_vs_operating_pct = (
    growth_investment /
    (growth_investment + operating_expense)
) * 100


growth_reinvestment = {
    "signal_name": "Growth Reinvestment",

    "observed_metrics": {
        "growth_investment": int(growth_investment),

        "growth_to_revenue_pct":
            round(float(growth_to_revenue_pct), 2),

        "growth_share_of_business_outflows_pct":
            round(float(growth_outflow_share_pct), 2),

        "growth_share_vs_operating_spend_pct":
            round(float(growth_vs_operating_pct), 2),

        "internal_savings_transfer":
            int(internal_transfer)
    },

    "positive_signal": growth_investment > 0,

    "interpretation": (
        "Elite Builds is directing part of its observed "
        "cash outflows toward technology, marketing, and "
        "capacity expansion rather than only routine expenses."
    ),

    "limitation": (
        "Growth classification is based on the transaction "
        "descriptions and categories supplied in the assignment."
    )
}


print("\nGrowth Reinvestment Signal")
print("--------------------------")

for key, value in growth_reinvestment.items():
    print(f"{key}: {value}")


Growth Reinvestment Signal
--------------------------
signal_name: Growth Reinvestment
observed_metrics: {'growth_investment': 10650, 'growth_to_revenue_pct': 9.51, 'growth_share_of_business_outflows_pct': 17.07, 'growth_share_vs_operating_spend_pct': 33.7, 'internal_savings_transfer': 5000}
positive_signal: True
interpretation: Elite Builds is directing part of its observed cash outflows toward technology, marketing, and capacity expansion rather than only routine expenses.
limitation: Growth classification is based on the transaction descriptions and categories supplied in the assignment.


In [12]:
# ---------------------------------
# STEP 8: ADDITIONAL POSITIVE SIGNALS
# ---------------------------------

# Revenue transactions
revenue_df = df[df["economic_bucket"] == "Revenue"]

revenue_transaction_count = len(revenue_df)

average_revenue_transaction = revenue_df["amount"].mean()

largest_revenue_transaction = revenue_df["amount"].max()

largest_revenue_share_pct = (
    largest_revenue_transaction / revenue
) * 100


# Savings / liquidity allocation
savings_to_revenue_pct = (
    internal_transfer / revenue
) * 100


revenue_breadth = {
    "signal_name": "Revenue Breadth",

    "observed_metrics": {
        "revenue_transactions": int(revenue_transaction_count),
        "average_revenue_transaction": round(
            float(average_revenue_transaction), 2
        ),
        "largest_revenue_transaction": int(
            largest_revenue_transaction
        ),
        "largest_revenue_share_pct": round(
            float(largest_revenue_share_pct), 2
        )
    },

    "positive_signal": revenue_transaction_count > 1,

    "interpretation": (
        "Observed revenue is supported by multiple client payments "
        "rather than a single transaction."
    )
}


liquidity_allocation = {
    "signal_name": "Liquidity Allocation",

    "observed_metrics": {
        "savings_transfer": int(internal_transfer),
        "savings_to_revenue_pct": round(
            float(savings_to_revenue_pct), 2
        ),
        "net_cash_movement": int(net_cash_movement)
    },

    "positive_signal": (
        internal_transfer > 0 and net_cash_movement > 0
    ),

    "interpretation": (
        "Elite Builds transferred funds to savings while the "
        "observed transaction period still showed positive net "
        "cash movement."
    ),

    "limitation": (
        "The transfer indicates liquidity allocation, but the "
        "provided data does not show the savings-account balance "
        "or broader liquidity position."
    )
}


print("\nRevenue Breadth Signal")
print("----------------------")
for key, value in revenue_breadth.items():
    print(f"{key}: {value}")


print("\nLiquidity Allocation Signal")
print("---------------------------")
for key, value in liquidity_allocation.items():
    print(f"{key}: {value}")


Revenue Breadth Signal
----------------------
signal_name: Revenue Breadth
observed_metrics: {'revenue_transactions': 4, 'average_revenue_transaction': 28000.0, 'largest_revenue_transaction': 45000, 'largest_revenue_share_pct': 40.18}
positive_signal: True
interpretation: Observed revenue is supported by multiple client payments rather than a single transaction.

Liquidity Allocation Signal
---------------------------
signal_name: Liquidity Allocation
observed_metrics: {'savings_transfer': 5000, 'savings_to_revenue_pct': 4.46, 'net_cash_movement': 44720}
positive_signal: True
interpretation: Elite Builds transferred funds to savings while the observed transaction period still showed positive net cash movement.
limitation: The transfer indicates liquidity allocation, but the provided data does not show the savings-account balance or broader liquidity position.


In [14]:
# ---------------------------------
# STEP 9: CREATE RAG-READY JSON
# ---------------------------------

import json

customer_segments = {
    "customer": "Elite Builds LLC",

    "demographic": {
        "industry": "Residential Construction",
        "location": "Utah",
        "business_size": "Small firm"
    },

    "behavioral": {
        "revenue_inflow": int(revenue),
        "material_spend": int(material_expense),
        "growth_investment": int(growth_investment),
        "net_cash_movement": int(net_cash_movement),
        "revenue_transactions": int(revenue_transaction_count),
        "savings_transfer": int(internal_transfer)
    },

    "psychographic": {
        "growth_orientation": "High",
        "technology_adoption": "Observed via AutoCAD spend",
        "customer_acquisition_focus": "Observed via Local SEO",
        "liquidity_discipline": "Suggested by savings transfer"
    },

    "positive_signals": [
        "Resilience Alpha",
        "Growth Reinvestment",
        "Revenue Breadth",
        "Liquidity Allocation"
    ],

    "market_context": {
        "material_cost_inflation_pct": 15,
        "industry_net_margin_pct": 4
    },

    "data_scope": "20 transactions, March 2026"
}


# Print formatted JSON
print("\nCustomer Segmentation JSON")
print("--------------------------")
print(json.dumps(customer_segments, indent=2))


# Save JSON file
with open(
    "customer_segments.json",
    "w"
) as f:
    json.dump(
        customer_segments,
        f,
        indent=2
    )


Customer Segmentation JSON
--------------------------
{
  "customer": "Elite Builds LLC",
  "demographic": {
    "industry": "Residential Construction",
    "location": "Utah",
    "business_size": "Small firm"
  },
  "behavioral": {
    "revenue_inflow": 112000,
    "material_spend": 30800,
    "growth_investment": 10650,
    "net_cash_movement": 44720,
    "revenue_transactions": 4,
    "savings_transfer": 5000
  },
  "psychographic": {
    "growth_orientation": "High",
    "technology_adoption": "Observed via AutoCAD spend",
    "customer_acquisition_focus": "Observed via Local SEO",
    "liquidity_discipline": "Suggested by savings transfer"
  },
  "positive_signals": [
    "Resilience Alpha",
    "Growth Reinvestment",
    "Revenue Breadth",
    "Liquidity Allocation"
  ],
  "market_context": {
    "material_cost_inflation_pct": 15,
    "industry_net_margin_pct": 4
  },
  "data_scope": "20 transactions, March 2026"
}


In [15]:
# ---------------------------------
# STEP 10: SYSTEM PROMPT
# ---------------------------------

SYSTEM_PROMPT = """
You are a Banking Growth Intelligence Analyst supporting a
Relationship Manager (RM).

Your task is to convert customer banking activity and supplied
market context into a concise, evidence-based customer segmentation
summary for use in a RAG system.

Instructions:

1. Focus on positive, commercially relevant signals without ignoring
   material risks or limitations.

2. Identify signals from the transaction data that indicate:
   - financial resilience,
   - growth reinvestment,
   - revenue behaviour,
   - liquidity behaviour.

3. Compare observed customer behaviour with the supplied market
   context where the data supports the comparison.

4. Create three customer segments:
   - demographic,
   - behavioral,
   - psychographic.

5. Demographic information must use only facts explicitly provided.

6. Psychographic attributes may be inferred from transaction behaviour,
   but clearly distinguish inference from observed fact.

7. Never invent customer facts, historical trends, financial ratios,
   creditworthiness, balances, or business characteristics that are not
   supported by the supplied data.

8. Do not treat internal transfers as operating expenses.

9. Do not describe transaction-based cash surplus as accounting profit
   or net margin.

10. Use a professional, concise, commercially useful tone appropriate
    for a bank Relationship Manager.

11. Return valid JSON only.

12. Keep the complete JSON response below 300 tokens.

Required JSON structure:

{
  "customer": "",
  "demographic": {},
  "behavioral": {},
  "psychographic": {},
  "positive_signals": [],
  "market_context": {},
  "data_scope": ""
}
"""

print("\nSYSTEM PROMPT")
print("-------------")
print(SYSTEM_PROMPT)


SYSTEM PROMPT
-------------

You are a Banking Growth Intelligence Analyst supporting a
Relationship Manager (RM).

Your task is to convert customer banking activity and supplied
market context into a concise, evidence-based customer segmentation
summary for use in a RAG system.

Instructions:

1. Focus on positive, commercially relevant signals without ignoring
   material risks or limitations.

2. Identify signals from the transaction data that indicate:
   - financial resilience,
   - growth reinvestment,
   - revenue behaviour,
   - liquidity behaviour.

3. Compare observed customer behaviour with the supplied market
   context where the data supports the comparison.

4. Create three customer segments:
   - demographic,
   - behavioral,
   - psychographic.

5. Demographic information must use only facts explicitly provided.

6. Psychographic attributes may be inferred from transaction behaviour,
   but clearly distinguish inference from observed fact.

7. Never invent customer fac

In [16]:
# ---------------------------------
# STEP 11: BANK PRODUCT RECOMMENDATION
# ---------------------------------

product_recommendation = {
    "recommended_product": "Equipment Financing / Equipment Term Loan",

    "reason": (
        "Elite Builds shows evidence of capacity expansion through "
        "a $10,000 new-equipment down payment and an existing "
        "excavator lease, while maintaining positive observed cash "
        "movement."
    ),

    "customer_benefit": (
        "Financing future equipment purchases could preserve "
        "working liquidity while allowing the business to continue "
        "expanding capacity during a period of elevated construction "
        "input costs."
    ),

    "supporting_signals": [
        "Growth Reinvestment",
        "Resilience Alpha",
        "Positive Net Cash Movement"
    ]
}


print("\nBank Product Recommendation")
print("---------------------------")

for key, value in product_recommendation.items():
    print(f"{key}: {value}")


Bank Product Recommendation
---------------------------
recommended_product: Equipment Financing / Equipment Term Loan
reason: Elite Builds shows evidence of capacity expansion through a $10,000 new-equipment down payment and an existing excavator lease, while maintaining positive observed cash movement.
customer_benefit: Financing future equipment purchases could preserve working liquidity while allowing the business to continue expanding capacity during a period of elevated construction input costs.
supporting_signals: ['Growth Reinvestment', 'Resilience Alpha', 'Positive Net Cash Movement']


In [17]:
# ---------------------------------
# STEP 12: RM HOOK
# ---------------------------------

rm_hook = (
    "Elite Builds appears to be maintaining strong project inflows "
    "while continuing to invest in technology and new equipment despite "
    "higher construction input costs. "
    "An equipment financing facility could help preserve working liquidity "
    "while supporting further capacity expansion."
)

print("\nRM Hook")
print("-------")
print(rm_hook)


RM Hook
-------
Elite Builds appears to be maintaining strong project inflows while continuing to invest in technology and new equipment despite higher construction input costs. An equipment financing facility could help preserve working liquidity while supporting further capacity expansion.


In [26]:
%%writefile scripts/signal_extraction.py

import json
from pathlib import Path
import pandas as pd


# =========================================================
# 1. PROJECT PATHS
# =========================================================

PROJECT_ROOT = Path(__file__).resolve().parents[1]

DATA_PATH = PROJECT_ROOT / "data" / "transactions.json"
OUTPUT_PATH = PROJECT_ROOT / "output" / "customer_segments.json"


# =========================================================
# 2. LOAD DATA
# =========================================================

df = pd.read_json(DATA_PATH)

print(f"Loaded {len(df)} transactions.")


# =========================================================
# 3. DATA QUALITY VALIDATION
# =========================================================

df["date"] = pd.to_datetime(df["date"])

missing_values = df.isnull().sum()
duplicate_count = df.duplicated().sum()

invalid_credits = df[
    (df["type"] == "Credit") &
    (df["amount"] <= 0)
]

invalid_debits = df[
    (df["type"] == "Debit") &
    (df["amount"] >= 0)
]

print("\nData Quality Validation")
print("-----------------------")
print("Missing Values:")
print(missing_values)

print(f"\nDuplicate Rows: {duplicate_count}")
print(f"Invalid Credits: {len(invalid_credits)}")
print(f"Invalid Debits: {len(invalid_debits)}")

print(
    f"Date Range: "
    f"{df['date'].min().date()} to "
    f"{df['date'].max().date()}"
)


# =========================================================
# 4. TRANSACTION PROFILING
# =========================================================

print("\nTransaction Type Count:")
print(df["type"].value_counts())

print("\nTransaction Count by Category:")
print(df["category"].value_counts())

category_summary = (
    df.groupby("category")
      .agg(
          transaction_count=("amount", "count"),
          net_amount=("amount", "sum")
      )
      .sort_values("net_amount", ascending=False)
)

print("\nCategory Summary:")
print(category_summary)


# =========================================================
# 5. ECONOMIC CLASSIFICATION
# =========================================================

category_mapping = {
    "Revenue": "Revenue",
    "Interest": "Other Income",
    "Inventory": "Material Expense",
    "Operations": "Operating Expense",
    "Equipment": "Operating Expense",
    "Tech/Growth": "Growth Investment",
    "Growth": "Growth Investment",
    "Savings": "Internal Transfer"
}

df["economic_bucket"] = df["category"].map(category_mapping)

if df["economic_bucket"].isnull().any():
    unknown_categories = df.loc[
        df["economic_bucket"].isnull(),
        "category"
    ].unique()

    raise ValueError(
        f"Unmapped categories found: {unknown_categories}"
    )


# =========================================================
# 6. CORE FINANCIAL METRICS
# =========================================================

revenue = df.loc[
    df["economic_bucket"] == "Revenue",
    "amount"
].sum()

other_income = df.loc[
    df["economic_bucket"] == "Other Income",
    "amount"
].sum()

material_expense = abs(
    df.loc[
        df["economic_bucket"] == "Material Expense",
        "amount"
    ].sum()
)

operating_expense = abs(
    df.loc[
        df["economic_bucket"] == "Operating Expense",
        "amount"
    ].sum()
)

growth_investment = abs(
    df.loc[
        df["economic_bucket"] == "Growth Investment",
        "amount"
    ].sum()
)

internal_transfer = abs(
    df.loc[
        df["economic_bucket"] == "Internal Transfer",
        "amount"
    ].sum()
)

core_operating_surplus = (
    revenue
    - material_expense
    - operating_expense
)

total_credits = df.loc[
    df["type"] == "Credit",
    "amount"
].sum()

total_debits = abs(
    df.loc[
        df["type"] == "Debit",
        "amount"
    ].sum()
)

net_cash_movement = total_credits - total_debits

material_to_revenue_pct = (
    material_expense / revenue
) * 100

operating_to_revenue_pct = (
    operating_expense / revenue
) * 100

growth_to_revenue_pct = (
    growth_investment / revenue
) * 100

core_cash_retention_pct = (
    core_operating_surplus / revenue
) * 100


# =========================================================
# 7. RESILIENCE ALPHA
# =========================================================

resilience_alpha = {
    "signal_name": "Resilience Alpha",
    "positive_signal": bool(
        net_cash_movement > 0 and growth_investment > 0
    ),
    "observed_metrics": {
        "revenue": int(revenue),
        "material_expense": int(material_expense),
        "material_burden_pct": round(
            float(material_to_revenue_pct), 2
        ),
        "core_operating_cash_surplus": int(
            core_operating_surplus
        ),
        "core_cash_retention_pct": round(
            float(core_cash_retention_pct), 2
        ),
        "net_cash_movement": int(net_cash_movement),
        "growth_investment": int(growth_investment)
    }
}


# =========================================================
# 8. GROWTH REINVESTMENT
# =========================================================

business_outflows = (
    material_expense
    + operating_expense
    + growth_investment
)

growth_outflow_share_pct = (
    growth_investment / business_outflows
) * 100

growth_vs_operating_pct = (
    growth_investment /
    (growth_investment + operating_expense)
) * 100

growth_reinvestment = {
    "signal_name": "Growth Reinvestment",
    "positive_signal": bool(growth_investment > 0),
    "observed_metrics": {
        "growth_investment": int(growth_investment),
        "growth_to_revenue_pct": round(
            float(growth_to_revenue_pct), 2
        ),
        "growth_share_of_business_outflows_pct": round(
            float(growth_outflow_share_pct), 2
        ),
        "growth_share_vs_operating_spend_pct": round(
            float(growth_vs_operating_pct), 2
        ),
        "internal_savings_transfer": int(internal_transfer)
    }
}


# =========================================================
# 9. REVENUE BREADTH
# =========================================================

revenue_df = df[
    df["economic_bucket"] == "Revenue"
]

revenue_transaction_count = len(revenue_df)

average_revenue_transaction = (
    revenue_df["amount"].mean()
)

largest_revenue_transaction = (
    revenue_df["amount"].max()
)

largest_revenue_share_pct = (
    largest_revenue_transaction / revenue
) * 100


# =========================================================
# 10. LIQUIDITY ALLOCATION
# =========================================================

savings_to_revenue_pct = (
    internal_transfer / revenue
) * 100


# =========================================================
# 11. FINAL RAG-READY JSON
# =========================================================

customer_segments = {
    "customer": "Elite Builds LLC",

    "demographic": {
        "industry": "Residential Construction",
        "location": "Utah",
        "business_size": "Small firm"
    },

    "behavioral": {
        "revenue_inflow": int(revenue),
        "material_spend": int(material_expense),
        "growth_investment": int(growth_investment),
        "net_cash_movement": int(net_cash_movement),
        "revenue_transactions": int(
            revenue_transaction_count
        ),
        "savings_transfer": int(internal_transfer)
    },

    "psychographic": {
        "growth_orientation": "High",
        "technology_adoption":
            "Observed via AutoCAD spend",
        "customer_acquisition_focus":
            "Observed via Local SEO",
        "liquidity_discipline":
            "Suggested by savings transfer"
    },

    "positive_signals": [
        "Resilience Alpha",
        "Growth Reinvestment",
        "Revenue Breadth",
        "Liquidity Allocation"
    ],

    "market_context": {
        "material_cost_inflation_pct": 15,
        "industry_net_margin_pct": 4
    },

    "data_scope":
        "20 transactions, March 2026"
}


# =========================================================
# 12. SAVE JSON
# =========================================================

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        customer_segments,
        f,
        indent=2
    )


print("\nCore Financial Metrics")
print("----------------------")
print(f"Revenue: ${revenue:,.2f}")
print(f"Material Expense: ${material_expense:,.2f}")
print(f"Operating Expense: ${operating_expense:,.2f}")
print(f"Growth Investment: ${growth_investment:,.2f}")
print(f"Net Cash Movement: ${net_cash_movement:,.2f}")

print("\nAnalysis completed successfully.")
print(f"JSON saved to: {OUTPUT_PATH}")

Writing scripts/signal_extraction.py


In [27]:
%run scripts/signal_extraction.py

Loaded 20 transactions.

Data Quality Validation
-----------------------
Missing Values:
date           0
description    0
category       0
amount         0
type           0
dtype: int64

Duplicate Rows: 0
Invalid Credits: 0
Invalid Debits: 0
Date Range: 2026-03-01 to 2026-03-31

Transaction Type Count:
type
Debit     15
Credit     5
Name: count, dtype: int64

Transaction Count by Category:
category
Operations     6
Revenue        4
Inventory      4
Tech/Growth    2
Equipment      1
Interest       1
Savings        1
Growth         1
Name: count, dtype: int64

Category Summary:
             transaction_count  net_amount
category                                  
Revenue                      4      112000
Interest                     1         120
Tech/Growth                  2        -650
Equipment                    1       -2200
Savings                      1       -5000
Growth                       1      -10000
Operations                   6      -18750
Inventory                    

In [28]:
import json
from pathlib import Path

json_path = Path("output/customer_segments.json")

# Check that the file exists
print("File exists:", json_path.exists())

# Load JSON
with open(json_path, "r", encoding="utf-8") as f:
    final_json = json.load(f)

# Display JSON
print("\nFinal RAG JSON:")
print(json.dumps(final_json, indent=2))

# Basic validation
required_sections = [
    "customer",
    "demographic",
    "behavioral",
    "psychographic",
    "positive_signals",
    "market_context",
    "data_scope"
]

missing_sections = [
    section
    for section in required_sections
    if section not in final_json
]

print("\nMissing required sections:", missing_sections)

# Count words as a quick size sanity check
json_text = json.dumps(final_json)

word_count = len(json_text.split())

print("Approximate word count:", word_count)
print("Character count:", len(json_text))

File exists: True

Final RAG JSON:
{
  "customer": "Elite Builds LLC",
  "demographic": {
    "industry": "Residential Construction",
    "location": "Utah",
    "business_size": "Small firm"
  },
  "behavioral": {
    "revenue_inflow": 112000,
    "material_spend": 30800,
    "growth_investment": 10650,
    "net_cash_movement": 44720,
    "revenue_transactions": 4,
    "savings_transfer": 5000
  },
  "psychographic": {
    "growth_orientation": "High",
    "technology_adoption": "Observed via AutoCAD spend",
    "customer_acquisition_focus": "Observed via Local SEO",
    "liquidity_discipline": "Suggested by savings transfer"
  },
  "positive_signals": [
    "Resilience Alpha",
    "Growth Reinvestment",
    "Revenue Breadth",
    "Liquidity Allocation"
  ],
  "market_context": {
    "material_cost_inflation_pct": 15,
    "industry_net_margin_pct": 4
  },
  "data_scope": "20 transactions, March 2026"
}

Missing required sections: []
Approximate word count: 63
Character count: 769


In [29]:
from pathlib import Path

project_root = Path(
    r"C:\Users\sneha\CR_Project\Sympera_construction_company_p1"
)

required_files = {
    "Input Data":
        project_root / "data" / "transactions.json",

    "Python Script":
        project_root / "scripts" / "signal_extraction.py",

    "RAG Output":
        project_root / "output" / "customer_segments.json",

    "README":
        project_root / "README.md"
}


print("FINAL PROJECT CHECK")
print("-------------------")

all_ok = True

for name, path in required_files.items():

    exists = path.exists()

    print(
        f"{name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    print(f"   {path}")

    if not exists:
        all_ok = False


print("\n-------------------")

if all_ok:
    print("PASS: All required assignment files are present.")
else:
    print("WARNING: Some required files are missing.")

FINAL PROJECT CHECK
-------------------
Input Data: FOUND
   C:\Users\sneha\CR_Project\Sympera_construction_company_p1\data\transactions.json
Python Script: FOUND
   C:\Users\sneha\CR_Project\Sympera_construction_company_p1\scripts\signal_extraction.py
RAG Output: FOUND
   C:\Users\sneha\CR_Project\Sympera_construction_company_p1\output\customer_segments.json
README: FOUND
   C:\Users\sneha\CR_Project\Sympera_construction_company_p1\README.md

-------------------
PASS: All required assignment files are present.


In [30]:
!git --version

'git' is not recognized as an internal or external command,
operable program or batch file.
